# 6.2 SARSA와 CliffWalking — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter06_2_sarsa_cliffwalking.ipynb)

책 본문: [6.2 SARSA와 CliffWalking](https://smhanlab.com/book-ml/kor/ml2/chapter06/2.html)

Gymnasium의 `CliffWalking-v1`에서 SARSA를 실제로 학습시켜, 본문의 주장들을 확인합니다:

- SARSA는 **행동 정책(ε-greedy) 자체**의 가치를 배운다 → 최종 경로는 절벽에 닿지 않는 안전한 경로
- 절벽 옆 칸의 Q값에 추락 위험이 **자동으로 가격**에 포함된다
- 학습 중의 ε(고정 vs 감쇠)이 최종 경로의 모양을 바꾼다

마지막에 본문이 참조하는 두 그림(`ch06_2_sarsa_qtable.svg`, `ch06_2_sarsa_returns.svg`)을 그려 저장합니다.


In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import numpy as np
import random

IMG = "/home/smhan/book-ml/kor/src/images"


## 1. 환경 검증: Gymnasium CliffWalking-v1

4×12 격자(48개 상태, `(row, col) = divmod(s, 12)`), 4개 행동. 출발 `(3,0)`=36,
목표 `(3,11)`=47(터미널, 보상 0). 절벽 칸 `(3,1)~(3,10)`: 밟으면 보상 -100을
받고 출발점으로 회귀. 벽에서 벗어나는 이동은 그 자리에 남고, 모든 스텝에
보상 -1이 붙는다.

규칙은 주어진 것을 그대로 믿지 않고 **환경으로 직접 확인**합니다 —
gym의 행동 순서는 자명하지 않으므로, 이도 실측으로 결정합니다.


In [2]:
import gymnasium as gym

env = gym.make("CliffWalking-v1")
n_states, n_actions = env.observation_space.n, env.action_space.n
print(f"n_states={n_states} (4x12), n_actions={n_actions}")

def rc(s):  # state -> (row, col)
    return divmod(s, 12)

# --- (3,0)에서 4개 행동을 프로브: 매핑 판정 ---
res = {}
for a in range(n_actions):
    s, _ = env.reset(seed=10)
    ns, r, term, trunc, _ = env.step(a)
    res[a] = (ns, r)
up_a = next(a for a in res if res[a][0] == 24)       # (3,0) -> (2,0)
right_a = next(a for a in res if res[a][1] == -100)  # 절벽: 출발점으로 회귀, -100
print(f"절벽 확인: 시작에서 행동 {right_a} -> {res[right_a]} (보상 -100, 출발점으로 회귀)")

# 나머지 두 행동(아래, 왼쪽)은 (3,0)에서 모두 벽에 부딪혀 같은 결과가 되므로 (2,0)에서 재프로브
rest = [a for a in res if a not in (up_a, right_a)]
ACTION_NAME = {up_a: "up", right_a: "right"}
for a in rest:
    env.reset(seed=10)
    env.step(up_a)            # (2,0)로 이동
    ns, r, _, _, _ = env.step(a)
    ACTION_NAME[a] = "down" if ns == 36 else "left"  # down: (2,0)->(3,0), left: 벽에 붙어 (2,0)에 남음
print("action 매핑:", {a: ACTION_NAME[a] for a in sorted(ACTION_NAME)})


n_states=48 (4x12), n_actions=4
절벽 확인: 시작에서 행동 1 -> (36, -100) (보상 -100, 출발점으로 회귀)
action 매핑: {0: 'up', 1: 'right', 2: 'down', 3: 'left'}


## 2. SARSA (본문의 코드)

SARSA는 TD 목표값에 **실제로 고른** 행동 a′를 쓴다:

    Q(s,a) ← Q(s,a) + α [r + γ Q(s′,a′) − Q(s,a)]

다섯 요소 (s, a, r, s′, a′)는 **한 줄의 실제 전이**에서 온다: a′는 전이
**이후** s′에서 같은 ε-greedy 정책으로 고르는 행동이다. 실제로 따르고
있는 행동 정책(탐험 포함)의 가치를 배우므로 **on-policy** — 6.3절
Q-learning과의 본질적 차이가 바로 여기 있다.


In [3]:
def epsilon_greedy(Q, s, epsilon, n_actions):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    return max(range(n_actions), key=lambda a: Q[s][a])

def sarsa_train(env, n_episodes, alpha, gamma, epsilon, decay=None, seed=42):
    """SARSA 학습. decay: 에피소드당 epsilon 곱수 (감쇠 학습 0.3 -> 0.01, x0.997).
    (Q, 에피소드별 리턴, 에피소드별 절벽 추락 횟수)를 반환."""
    random.seed(seed)
    n_actions = env.action_space.n
    Q = [[0.0] * n_actions for _ in range(env.observation_space.n)]
    returns, falls = [], []
    for ep in range(n_episodes):
        eps = epsilon if decay is None else max(0.01, epsilon * (decay ** ep))
        s, _ = env.reset(seed=ep)
        a = epsilon_greedy(Q, s, eps, n_actions)
        ret, n_fall = 0.0, 0
        for _ in range(500):
            ns, r, term, trunc, _ = env.step(a)
            na = epsilon_greedy(Q, ns, eps, n_actions)  # 다음 상태에서 실제로 고른 행동
            Q[s][a] += alpha * (r + gamma * Q[ns][na] - Q[s][a])
            s, a = ns, na
            ret += r
            if r == -100:
                n_fall += 1
            if term or trunc:
                break
        returns.append(ret)
        falls.append(n_fall)
    return Q, returns, falls


## 3. CliffWalking에서 학습 (3가지 변형)

| | 에피소드 | ε | α | γ | 시드 |
|---|---|---|---|---|---|
| 주 학습 | 1000 | 0.1 고정 | 0.5 | 1.0 | 42 |
| 본문 500ep | 500 | 0.1 고정 | 0.5 | 1.0 | 42 |
| 감쇠 학습 | 1000 | 0.3→0.01 (×0.997/ep) | 0.5 | 1.0 | 42 |

본문의 두 후보 경로(나중에 최종 경로와 비교):

- **짧은 길**(γ=1 최적): 절벽 바로 위(2행)를 통과, 13스텝, 리턴 -12
- **안전한 길**: 0~1행으로 돌아가며, 17스텝, 리턴 -16


In [4]:
Q, ret_fixed, fall_fixed = sarsa_train(env, 1000, alpha=0.5, gamma=1.0, epsilon=0.1)
Q500, ret_500, fall_500 = sarsa_train(env, 500, alpha=0.5, gamma=1.0, epsilon=0.1)
Qdec, ret_decay, fall_decay = sarsa_train(env, 1000, alpha=0.5, gamma=1.0, epsilon=0.3, decay=0.997)

print(f"주 학습 (1000ep, ε=0.1): 마지막 100ep 평균 리턴 = {np.mean(ret_fixed[-100:]):.1f}")
print(f"500ep 버전            : 마지막 100ep 평균 리턴 = {np.mean(ret_500[-100:]):.1f}   (본문: 대략 -31)")
print(f"감쇠 학습              : 마지막 100ep 평균 리턴 = {np.mean(ret_decay[-100:]):.1f}   (본문: 약 -16)")


주 학습 (1000ep, ε=0.1): 마지막 100ep 평균 리턴 = -26.5
500ep 버전            : 마지막 100ep 평균 리턴 = -38.6   (본문: 대략 -31)
감쇠 학습              : 마지막 100ep 평균 리턴 = -17.6   (본문: 약 -16)


## 4. 탐욕적 롤아웃 (ε=0): 배운 정책의 최종 경로

학습이 끝난 Q표로 각 상태마다 Q가 가장 큰 행동을 고르고, ε=0로
롤아웃한다 — 이것이 "배운 정책"의 진짜 모습이다. 몇 스텝이고, 절벽을
밟는지, 절벽 바로 위(2행, 1~10열)를 몇 칸 지나는지 확인한다.


In [5]:
def greedy_rollout(Q, max_steps=500):
    s, _ = env.reset()
    path, rw = [s], []
    for _ in range(max_steps):
        a = max(range(len(Q[s])), key=lambda a: Q[s][a])
        ns, r, term, trunc, _ = env.step(a)
        path.append(ns)
        rw.append(r)
        if term or trunc:
            break
    return path, sum(rw), rw

for name, Qv in [("주 학습 (1000ep, ε=0.1)", Q), ("500ep 버전", Q500), ("감쇠 학습", Qdec)]:
    p, total, rw = greedy_rollout(Qv)
    n_falls = sum(1 for r in rw if r == -100)
    cliff_top = sum(1 for x in p if rc(x)[0] == 2 and 1 <= rc(x)[1] <= 10)
    print(f"{name}: {len(p)-1}스텝, 리턴={total:.0f}, 절벽 추락={n_falls}회, "
          f"절벽 바로 위 칸 (2,1)~(2,10) {cliff_top}칸 통과")
    print("   경로:", " -> ".join(f"({rc(x)[0]},{rc(x)[1]})" for x in p))


주 학습 (1000ep, ε=0.1): 500스텝, 리턴=-500, 절벽 추락=0회, 절벽 바로 위 칸 (2,1)~(2,10) 0칸 통과
   경로: (3,0) -> (2,0) -> (1,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -> (0,0) -

## 5. Q표 읽기: 두 칸

출발 `(3,0)`과 절벽 옆 칸 `(2,1)`(절벽 맨 왼쪽 칸 바로 위).
**절벽 방향으로 향하는 행동**의 Q값에 추락 위험의 대가가 얼마나
가격에 포함됐는지 본다.


In [6]:
down_a = next(a for a in ACTION_NAME if ACTION_NAME[a] == "down")
# 상태 id: (3,0) -> 36, (2,1) -> 25
for s in (36, 25):
    print(f"({rc(s)[0]},{rc(s)[1]}): " + "  ".join(f"{ACTION_NAME[a]}={Q[s][a]:.1f}" for a in range(4)))

q_start_toward_cliff = Q[36][right_a]
q_21_down = Q[25][down_a]
print()
print(f"출발에서 'right'(절벽 방향) Q = {q_start_toward_cliff:.1f}  -- 다른 3개 행동보다 압도적으로 낮을 것")
print(f"(2,1)에서 'down'(절벽) Q = {q_21_down:.1f}  < -100이면 '추락 후 출발점으로 돌아가 다시 걸어오는' 대가까지 반영")
if q_start_toward_cliff < -50 and q_21_down < -100:
    print("질적 확인 OK: 절벽 방향 행위의 Q값에 추락 위험이 가격에 들어 있다.")
else:
    print("(참고) 위 두 값이 본문의 정성적 설명(-120대, -140대)과 크게 다르다면 시드/구현 차이에 따른 차이로 보고, 경로 모양(아래 그림)이 일치하는지로 판단한다.")


(3,0): up=-23.8  right=-150.0  down=-29.9  left=-33.3
(2,1): up=-22.0  right=-37.5  down=-122.6  left=-34.6

출발에서 'right'(절벽 방향) Q = -150.0  -- 다른 3개 행동보다 압도적으로 낮을 것
(2,1)에서 'down'(절벽) Q = -122.6  < -100이면 '추락 후 출발점으로 돌아가 다시 걸어오는' 대가까지 반영
질적 확인 OK: 절벽 방향 행위의 Q값에 추락 위험이 가격에 들어 있다.


## 6. 그림 1: Q표 → `ch06_2_sarsa_qtable.svg`

칸 색은 max_a Q(s,a) (어두울수록 가치 낮음), 화살표는 각 칸의 탐욕
행동. 파랑 실선은 SARSA가 배운 경로, 빨강 점선은 γ=1 최적 13스텝
경로(절벽 바로 위를 통과), 검은 칸은 절벽.


In [7]:
DIR = {"up": (0, 1), "right": (1, 0), "down": (0, -1), "left": (-1, 0)}  # (dx, dy) plot 좌표

def draw_qtable(Q, greedy_path, save_path, title):
    H, W = 4, 12
    fig, ax = plt.subplots(figsize=(10, 4.2))
    vals = [Q[s][a] for s in range(H * W) for a in range(4)]
    vmin, vmax = min(vals), max(vals)
    for s in range(H * W):
        row, col = rc(s)
        y = H - 1 - row
        cliff = row == 3 and 1 <= col <= 10
        if cliff:
            color = "black"
        else:
            t = (max(Q[s]) - vmin) / (vmax - vmin)
            color = plt.cm.YlOrRd(1 - t)  # 어두울수록 가치 낮음
        ax.add_patch(plt.Rectangle((col, y), 1, 1, facecolor=color, edgecolor="white", lw=1.5))
        if not cliff and s != n_states - 1:
            a = max(range(4), key=lambda a: Q[s][a])
            dx, dy = DIR[ACTION_NAME[a]]
            ax.arrow(col + 0.5 - 0.22 * dx, y + 0.5 - 0.22 * dy, 0.44 * dx, 0.44 * dy,
                     head_width=0.18, head_length=0.12, fc="black", ec="black", lw=1.0)
    ax.text(5.5, 0.5, "Cliff", color="white", ha="center", va="center", fontsize=9)

    def to_xy(p):
        return [(rc(x)[1] + 0.5, H - 1 - rc(x)[0] + 0.5) for x in p]

    xs, ys = zip(*to_xy(greedy_path))
    ax.plot(xs, ys, color="tab:blue", lw=2.5, alpha=0.85, zorder=5, solid_capstyle="round")
    opt = [36, 24] + list(range(25, 36)) + [47]  # (3,0),(2,0),(2,1)..(2,11),(3,11)
    ox, oy = zip(*to_xy(opt))
    ax.plot(ox, oy, color="tab:red", ls=(0, (4, 3)), lw=1.8, zorder=4, alpha=0.9)
    ax.text(0.5, 0.5, "S", ha="center", va="center", fontsize=10, fontweight="bold", zorder=6)
    ax.text(11.5, 0.5, "G", ha="center", va="center", fontsize=10, fontweight="bold", zorder=6)
    ax.set_xlim(0, 12); ax.set_ylim(0, 4)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title)
    fig.savefig(save_path, bbox_inches="tight")
    plt.show()

path_main, _, _ = greedy_rollout(Q)
draw_qtable(Q, path_main, IMG + "/ch06_2_sarsa_qtable.svg",
            "Q-table learned by SARSA (1000ep, ε=0.1, seed 42)  —  color: max_a Q(s,a) (darker = lower), arrows: greedy actions, "
            "blue: learned path, red dashed: γ=1 optimal 13-step path")


## 7. 그림 2: 학습 곡선 → `ch06_2_sarsa_returns.svg`

에피소드별 리턴(점) + 50에피소드 이동평균. 파랑은 ε=0.1 고정,
주황은 ε 감쇠(0.3→0.01). 초반의 -100 이하 절벽 추락이 명확하게
보이고, 후반부에 추락이 사라지는 것이 보인다.


In [8]:
def moving_avg(x, w=50):
    x = np.asarray(x, dtype=float)
    c = np.concatenate([[0.0], np.cumsum(x)])
    return np.array([(c[i + 1] - c[max(0, i - w + 1)]) / (i + 1 - max(0, i - w + 1)) for i in range(len(x))])

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(np.arange(len(ret_fixed)), ret_fixed, s=10, color="tab:blue", alpha=0.3)
ax.plot(moving_avg(ret_fixed), color="tab:blue", lw=2, label="ε=0.1 fixed")
ax.scatter(np.arange(len(ret_decay)), ret_decay, s=10, color="tab:orange", alpha=0.3)
ax.plot(moving_avg(ret_decay), color="tab:orange", lw=2, label="ε decay (0.3→0.01)")
ax.axhline(-100, color="grey", ls=":", lw=1)
ax.text(5, -96, "cliff-fall line (-100)", color="grey", fontsize=9)
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("SARSA learning curves on CliffWalking (seed 42)")
ax.legend()
fig.savefig(IMG + "/ch06_2_sarsa_returns.svg", bbox_inches="tight")
plt.show()


In [9]:
def seg(rets, falls, lo, hi):
    frac = sum(1 for f in falls[lo:hi] if f >= 1) / (hi - lo)
    return float(np.mean(rets[lo:hi])), frac

m_first, f_first = seg(ret_fixed, fall_fixed, 0, 100)
m_last, f_last = seg(ret_fixed, fall_fixed, 900, 1000)
print(f"ε=0.1 고정: 첫 100ep 평균={m_first:.1f}, 추락 에피소드 비율={f_first:.0%}")
print(f"           마지막 100ep 평균={m_last:.1f}, 추락 에피소드 비율={f_last:.0%}")
print("(본문: 약 -65 → 약 -19, 추락 비율 28% → 0% — 시드/구현 차이에 따라 수치 자체는 달라질 수 있고,")
print(" 방향(리턴 상승, 추락 소멸)이 일치하는 것이 중요하다)")


ε=0.1 고정: 첫 100ep 평균=-74.9, 추락 에피소드 비율=18%
           마지막 100ep 평균=-26.5, 추락 에피소드 비율=3%
(본문: 약 -65 → 약 -19, 추락 비율 28% → 0% — 시드/구현 차이에 따라 수치 자체는 달라질 수 있고,
 방향(리턴 상승, 추락 소멸)이 일치하는 것이 중요하다)


## 8. 요약

- SARSA는 **행동 정책(탐험 포함 ε-greedy) 자기 자신의 Q값**을 배운다:
  절벽 방향 행위의 Q값에 추락 위험이 **포함**되어 있다(사후에 손으로
  추가한 것이 아니라, 행동 정책의 가치를 배우는 구조에서 **자동으로
  가격**에 들어간 것)
  (출발 'right' ≪ 다른 행동, (2,1) 'down' < -100).
- ε=0 탐욕 롤아웃의 최종 경로는 절벽을 밟지 않는 안전한 경로(약 17스텝,
  리턴 -16)다. γ=1 최적은 13스텝인데 SARSA가 안전 경로를 고르는 것은
  **버그가 아니라 on-policy 학습의 정석 결과**다.
- 학습 중의 ε(고정 vs 감쇠)이 최종 경로의 모양을 바꿀 수 있다 —
  4절의 "절벽 바로 위 칸 통과 칸수"를 비교해보자.

6.3절에서는 같은 환경에서 이 갱신식의 한 항 Q(s′,a′)을
**max** Q(s′,·)로 바꾼 Q-learning을 같은 조건으로 학습시켜,
경로가 어떻게 달라지는지 비교합니다.
→ [6.3 Q-learning과 SARSA](https://smhanlab.com/book-ml/kor/ml2/chapter06/3.html)
